# Trajectory collection on GPURuns `collect.py` over EvalPlus problems with Qwen2.5-1.5B-Instruct and writeslabeled trajectories back to Google Drive.**Why Drive:** Colab sessions disconnect after a few hours. Output is written toDrive as each attempt completes, so a disconnect costs at most one trajectory.To resume, just re-run the collection cell — finished work is skippedautomatically.**Order:** run every cell top to bottom. Cell 5 is a short smoke test; only runcell 6 (the full collection) once cell 5 looks right.

## 1. Confirm a GPU is attachedIf this prints `cpu`, use Runtime > Change runtime type > T4 GPU, then rerun.

In [ ]:
import torch

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then rerun.")

## 2. Mount DriveThis is where the dataset lives so it survives disconnects.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import pathlib

OUT_DIR = pathlib.Path("/content/drive/MyDrive/mech-interp/data/raw")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("output dir:", OUT_DIR)

## 3. Get the codeClones on a fresh session, pulls if the repo is already there.

In [ ]:
%%bash
cd /content
if [ -d mech-interp/.git ]; then
  cd mech-interp && git pull --ff-only
else
  git clone https://github.com/oakeys3/mech-interp.git
fi

## 4. Install dependencies`transformers>=5` matters: `model.py` uses the 5.x `dtype=` argument and the dict-returning `apply_chat_template`.

In [ ]:
%%bash
pip install -q "transformers>=5.0" evalplus

## 5. Smoke test — 3 problemsProves generation, grading, and labeling all work here before committing to thefull run. Expect a few minutes, mostly model download.

In [ ]:
%cd /content/mech-interp
!python -m data.collection.collect     --subset humaneval     --limit 3     --out "/content/drive/MyDrive/mech-interp/data/raw/smoke.jsonl"

In [ ]:
!python -m data.collection.label     "/content/drive/MyDrive/mech-interp/data/raw/smoke.jsonl"     --out "/content/drive/MyDrive/mech-interp/data/processed/smoke_labeled.jsonl"

## 6. Full collectionAll 164 HumanEval+ problems. Rerun this exact cell after any disconnect — itresumes rather than restarting.Raise `--samples-per-problem` (with `--temperature` above 0) if the class countsin step 7 come up short of 50 per failure class.

In [ ]:
%cd /content/mech-interp
!python -m data.collection.collect     --subset humaneval     --temperature 0.2     --samples-per-problem 1     --correction-rounds 3     --out "/content/drive/MyDrive/mech-interp/data/raw/humaneval_trajectories.jsonl"

## 7. Label and check the class distributionThis is the Phase 1 exit criterion: at least 50 examples per failure class.

In [ ]:
%cd /content/mech-interp
!python -m data.collection.label     "/content/drive/MyDrive/mech-interp/data/raw/humaneval_trajectories.jsonl"     --out "/content/drive/MyDrive/mech-interp/data/processed/humaneval_labeled.jsonl"

## 8. If MBPP+ is neededHumanEval+ alone yields at most 164 trajectories. MBPP+ adds ~378 more, which isthe straightforward way to reach the 500-800 target.

In [ ]:
%cd /content/mech-interp
!python -m data.collection.collect     --subset mbpp     --temperature 0.2     --correction-rounds 3     --out "/content/drive/MyDrive/mech-interp/data/raw/mbpp_trajectories.jsonl"